In [15]:
import pandas as pd
from pathlib import Path

In [16]:
RAW_FILE = "../data/raw/ontario_demand_2025.csv"

df = pd.read_csv(RAW_FILE, skiprows=3)
df.head()

,Date,Hour,Market Demand,Ontario Demand
0,2025-01-01,1,17247,13887
1,2025-01-01,2,17355,13722
2,2025-01-01,3,17638,13688
3,2025-01-01,4,17065,13613
4,2025-01-01,5,17498,13593


In [17]:
df = df.rename(columns={
    "Date": "date",
    "Hour": "hour",
    "Market Demand": "market_demand",
    "Ontario Demand": "ontario_demand"
})

df.columns

Index(['date', 'hour', 'market_demand', 'ontario_demand'], dtype='str')

In [18]:
df["date"] = pd.to_datetime(df["date"])
df["hour"] = pd.to_numeric(df["hour"], errors="raise")
df["market_demand"] = pd.to_numeric(df["market_demand"], errors="raise")
df["ontario_demand"] = pd.to_numeric(df["ontario_demand"], errors="raise")

In [19]:
df.dtypes

date              datetime64[us]
hour                       int64
market_demand              int64
ontario_demand             int64
dtype: object

In [20]:
df["timestamp"] = (
    df["date"]
    + pd.to_timedelta(df["hour"] - 1, unit="h")
)
df[["date", "hour", "timestamp"]].head()

,date,hour,timestamp
0,2025-01-01,1,2025-01-01 00:00:00
1,2025-01-01,2,2025-01-01 01:00:00
2,2025-01-01,3,2025-01-01 02:00:00
3,2025-01-01,4,2025-01-01 03:00:00
4,2025-01-01,5,2025-01-01 04:00:00


In [21]:
df[["date", "hour", "timestamp"]].head(24).tail()

,date,hour,timestamp
19,2025-01-01,20,2025-01-01 19:00:00
20,2025-01-01,21,2025-01-01 20:00:00
21,2025-01-01,22,2025-01-01 21:00:00
22,2025-01-01,23,2025-01-01 22:00:00
23,2025-01-01,24,2025-01-01 23:00:00


In [22]:
df = df.sort_values("timestamp").reset_index(drop=True)
df.head()

,date,hour,market_demand,ontario_demand,timestamp
0,2025-01-01,1,17247,13887,2025-01-01 00:00:00
1,2025-01-01,2,17355,13722,2025-01-01 01:00:00
2,2025-01-01,3,17638,13688,2025-01-01 02:00:00
3,2025-01-01,4,17065,13613,2025-01-01 03:00:00
4,2025-01-01,5,17498,13593,2025-01-01 04:00:00


In [23]:
df.tail()

,date,hour,market_demand,ontario_demand,timestamp
8754,2025-12-31,20,22400,18783,2025-12-31 19:00:00
8755,2025-12-31,21,21817,18294,2025-12-31 20:00:00
8756,2025-12-31,22,21290,17789,2025-12-31 21:00:00
8757,2025-12-31,23,20515,17415,2025-12-31 22:00:00
8758,2025-12-31,24,20042,16946,2025-12-31 23:00:00


In [24]:
print("Rows:", len(df))
print("Missing values:")
print(df.isna().sum())
print("\nDuplicates:")
print(df.duplicated(subset=["timestamp"]).sum())
print("\nHour range:")
print(df["hour"].min(), "to", df["hour"].max())
print("\nDate range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

Rows: 8759
Missing values:
date              0
hour              0
market_demand     0
ontario_demand    0
timestamp         0
dtype: int64

Duplicates:
0

Hour range:
1 to 24

Date range:
2025-01-01 00:00:00 to 2025-12-31 23:00:00


In [25]:
df = df[
    [
        "timestamp",
        "date",
        "hour",
        "market_demand",
        "ontario_demand"
    ]
]

df.head()

,timestamp,date,hour,market_demand,ontario_demand
0,2025-01-01 00:00:00,2025-01-01,1,17247,13887
1,2025-01-01 01:00:00,2025-01-01,2,17355,13722
2,2025-01-01 02:00:00,2025-01-01,3,17638,13688
3,2025-01-01 03:00:00,2025-01-01,4,17065,13613
4,2025-01-01 04:00:00,2025-01-01,5,17498,13593


In [26]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "ontario_demand_2025_clean.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

Saved 8759 rows to ..\data\processed\ontario_demand_2025_clean.csv


In [27]:
print("Rows:", len(df))
print(df.isna().sum())
print("Duplicates:", df.duplicated(subset=["timestamp"]).sum())
print(df["timestamp"].min(), df["timestamp"].max())

Rows: 8759
timestamp         0
date              0
hour              0
market_demand     0
ontario_demand    0
dtype: int64
Duplicates: 0
2025-01-01 00:00:00 2025-12-31 23:00:00


In [28]:
assert df["timestamp"].isna().sum() == 0
assert df["ontario_demand"].isna().sum() == 0
assert df["timestamp"].duplicated().sum() == 0
assert df["hour"].between(1, 24).all()
print("All validation checks passed")

All validation checks passed
